In [1]:
!pip install -q transformers peft trl datasets accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 48.2 MB/s eta 0:00:00


In [2]:
from google.colab import files
uploaded = files.upload()  # Chọn file qa_pairs.jsonl từ máy tính
print('✓ Upload xong:', list(uploaded.keys()))

Saving qa_pairs.jsonl to qa_pairs.jsonl
✓ Upload xong: ['qa_pairs.jsonl']


In [3]:
from datasets import load_dataset
import json

dataset = load_dataset('json', data_files='qa_pairs.jsonl', split='train')
print(f'Tổng số cặp Q&A: {len(dataset)}')
print('Ví dụ:')
print('  Câu hỏi:', dataset[0]['instruction'])
print('  Trả lời:', dataset[0]['output'][:200])

Generating train split: 0 examples [00:00, ? examples/s]

Tổng số cặp Q&A: 313
Ví dụ:
  Câu hỏi: Ai dự cuộc gặp không chính thức?
  Trả lời: Phan Văn Khải


In [4]:
def format_example(example):
    text = (
        f"### Câu hỏi:\n{example['instruction']}\n\n"
        f"### Trả lời:\n{example['output']}\n"
    )
    return {'text': text}

dataset = dataset.map(format_example)
print('Ví dụ sau format:')
print(dataset[0]['text'][:300])

Map:   0%|          | 0/313 [00:00<?, ? examples/s]

Ví dụ sau format:
### Câu hỏi:
Ai dự cuộc gặp không chính thức?

### Trả lời:
Phan Văn Khải



In [5]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = 'SeaLLMs/SeaLLMs-v3-7B-Chat'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f'Đang load {model_name}...')
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)


Đang load SeaLLMs/SeaLLMs-v3-7B-Chat...


config.json:   0%|          | 0.00/811 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/367 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/211 [00:00<?, ?B/s]

In [7]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 10,092,544 || all params: 7,625,709,056 || trainable%: 0.1323


In [8]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir='./lora_output',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=50,
    lr_scheduler_type='cosine',
    logging_steps=20,
    save_strategy='epoch',
    bf16=True,
    optim='paged_adamw_8bit',
    report_to='none',
    gradient_checkpointing=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

trainer.train()


Adding EOS to train dataset:   0%|          | 0/313 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/313 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None, 'pad_token_id': 151643}.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
20,2.497136
40,1.424750
60,1.107179


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=60, training_loss=1.6763548215230306, metrics={'train_runtime': 117.1687, 'train_samples_per_second': 8.014, 'train_steps_per_second': 0.512, 'total_flos': 1896242910750720.0, 'train_loss': 1.6763548215230306})

In [9]:
trainer.model.save_pretrained('./lora_adapter')
tokenizer.save_pretrained('./lora_adapter')
print('✓ Đã lưu adapter tại ./lora_adapter/')

✓ Đã lưu adapter tại ./lora_adapter/


In [10]:
def ask(question: str, max_new_tokens: int = 300):
    messages = [
        {"role": "system", "content": "Bạn là trợ lý lịch sử chuyên về lịch sử Việt Nam. Trả lời bằng tiếng Việt."},
        {"role": "user", "content": question}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.3,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return text.strip()



In [11]:
from peft import PeftModel

# model = model.merge_and_unload()
model.eval()

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear4bit(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear4bit(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm

In [13]:
print(ask('Nước việt được ra đời năm bao nhiêu ?'))

Theo các tài liệu và nghiên cứu, nước Việt có nguồn gốc từ thời kỳ Hùng Vương vào khoảng thế kỷ thứ 3 TCN (Trước Công nguyên). Đây là giai đoạn hình thành nền văn hóa Đông Sơn với những di tích khảo cổ học quan trọng như đền Thăng Long - Hà Nội ngày nay.

Hưng Đạo Vương Lạc Tôn Thiên Hoàng đã lập nên vương quốc Âu Lạc sau đó cho đến khi An Dương Vương xây dựng kinh đô ở Văn Lang trên sông Hồng thì mới chính thức đánh dấu sự tồn tại của một nhà nước độc lập. Việc này xảy ra trong khoảng giữa thế kỷ thứ nhất trước công nguyên.
Vậy nên nếu nói cụ thể hơn: Nước Việt được tạo lập ngay từ rất sớm nhưng thực sự trở thành đất nước tự chủ, không phụ thuộc dưới bất kì quyền lực nào khác kể từ cuộc kháng chiến chống quân xâm lược Mông Nguyên lần đầu tiên diễn ra từ cuối thế kỷ XIII tới nửa đầu thế kỷ XIV. Nhưng cũng phải nhắc lại rằng tuy vậy, việc tuyên bố "Việt nam" chỉ bắt đầu từ triều Nguyễn do nhu cầu phát triển xã hội dân tộc trong quá trình giao lưu văn hoá cùng phương Tây.


In [15]:
%pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=a975c925bca8755abae0bbee7ac45b49fa057108ead7df5a57cfa05d09f63375
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [18]:
import shutil
from google.colab import files
shutil.make_archive('lora_adapter', 'zip', '.', 'lora_adapter')
files.download('lora_adapter.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# from peft import PeftModel

# # model = model.merge_and_unload()
# model.eval()

In [22]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch

# Load model + adapter
print("Đang load model...")
model_name = 'SeaLLMs/SeaLLMs-v3-7B-Chat'
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained('./lora_adapter')
base = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map='auto')
model = PeftModel.from_pretrained(base, './lora_adapter')
model.eval()
print("✓ Sẵn sàng!\n")

# Hàm trả lời
def ask(question: str, max_new_tokens: int = 300):
    messages = [
        {"role": "system", "content": "Bạn là trợ lý pháp luật chuyên về Bộ Luật Dân Sự Việt Nam 2015. Trả lời bằng tiếng Việt."},
        {"role": "user", "content": question}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.3,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return text.strip()

# Chat loop
print("=" * 50)
print("  CHATBOT Lịch sử")
print("  Gõ 'quit' để thoát")
print("=" * 50)

while True:
    question = input("\nCâu hỏi: ").strip()
    if question.lower() == 'quit':
        print("Tạm biệt!")
        break
    if not question:
        continue
    print(f"\nTrả lời: {ask(question)}")


Đang load model...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

✓ Sẵn sàng!

  CHATBOT Lịch sử
  Gõ 'quit' để thoát

Câu hỏi: Kim ngạch nhập khẩu từ Indonesia vào năm 2000 là bao nhiêu?

Trả lời: 7,86 tỷ USD

Câu hỏi: quit
Tạm biệt!
